# Step 4. Transfer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/4_transfer.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**So far.** Step 2 made the trial before a factor, and step 3 made blocks that are mostly congruent or mostly incongruent. This step puts both ideas together and adds a second task.

**The question.** Does adapting to frequent conflict make it harder to switch to another task?

**Prediction if shielding and switching are one control setting.** The switch cost is larger in a mostly incongruent context than in a mostly congruent one.

**The alternative to rule out.** A mostly incongruent context is simply harder, so everything is slower there.

**The analysis.** The switch cost (switch against repeat trials) in mostly incongruent mini-blocks against mostly congruent mini-blocks.

### As before, plus a second task

A cue says on each trial whether to name the ink or read the word. The task is a regular factor.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says
task = Factor("task", ["name the colour", "read the word"])        # new: a second task, with a cue

### New: the task transition

A transition factor, as in step 2, but on the task: does it repeat or switch? Without repeat trials in the same context there is no switch cost to measure.

In [ ]:
def task_repeats(task):
    return task[0] == task[-1]


def task_switches(task):
    return task[0] != task[-1]


transition = Factor("transition", [
    DerivedLevel("repeat", Transition(task_repeats, [task])),
    DerivedLevel("switch", Transition(task_switches, [task])),
])

### Something has to give

The blocks of step 3 hold 24 trials. Crossed with 2 tasks and 2 transitions that is 96 trials per mini-block: too long for a context that should change often.

So this design crosses what the analysis compares, congruency, the task and the transition, and gives up the even pairs. Congruency gets a weight instead of copies: the third argument of a level. 3 to 1 is three quarters. That is 16 combinations, plus a first trial.

In [ ]:
WEIGHTS = {"mostly congruent": (3, 1), "mostly incongruent": (1, 3)}     # (congruent, incongruent)


def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word


def mini_block(context):
    congruent_weight, incongruent_weight = WEIGHTS[context]
    congruency = Factor("congruency", [
        DerivedLevel("congruent", WithinTrial(same, [colour, word]), congruent_weight),
        DerivedLevel("incongruent", WithinTrial(different, [colour, word]), incongruent_weight),
    ])
    return CrossBlock(design=[colour, word, task, congruency, transition],
                      crossing=[congruency, task, transition],
                      constraints=[MinimumTrials(17)])

### A session of four mini-blocks

In [ ]:
blocks = []
for number, context in enumerate(["mostly congruent", "mostly incongruent", "mostly incongruent", "mostly congruent"], start=1):
    trials = pd.DataFrame(synthesize_trials(mini_block(context), 1, CMSGen)[0])
    trials.insert(0, "context", context)
    trials.insert(0, "block", number)
    blocks.append(trials)
session = pd.concat(blocks, ignore_index=True)
counted = session[session["transition"] != ""]       # the first trial of a mini-block has no transition
display(pd.crosstab(counted["context"], [counted["transition"], counted["congruency"]]))

transition            repeat                switch            
congruency         congruent incongruent congruent incongruent
context                                                       
mostly congruent          12           4        12           4
mostly incongruent         4          12         4          12

### What gave way?

The ink colour and the pairs are no longer in the crossing. Count them.

In [ ]:
display(pd.crosstab(counted["context"], counted["colour"]))
display(pd.crosstab([counted["context"], counted["colour"]], counted["word"]))

colour,blue,green,red
context,,,
mostly congruent,12,9,11
mostly incongruent,7,13,12


word                       blue  green  red
context            colour                  
mostly congruent   blue      10      1    1
                   green      2      6    1
                   red        1      2    8
mostly incongruent blue       3      2    2
                   green      3      4    6
                   red        5      6    1

**Check.** Is there a repeat baseline in each context? How many incongruent switch trials does a mostly congruent mini-block have? Are the ink colours and the pairs still even, and does that matter for the switch cost?

**The idea.** Each question added a factor, and each factor multiplies the design. At some point not everything fits, and the design becomes a choice about what to leave open. Which choice is acceptable depends on what you compare.